# 01 · Ingestion — what is in our knowledge base?

**Step 1 of the Study Assistant project.** This notebook loads every course file and checks that the text we extracted is complete and clean *before* we build any retrieval on top of it (garbage in → garbage out).

What the ingestion does (code in `src/study_assistant/ingest/`):

| Source | Unit (one `Document`) | Notes |
|---|---|---|
| Lecture slides `.pptx` | one slide | title, text boxes, tables, speaker notes; slide-number fields removed |
| Practical / lab notebooks `.ipynb` | one *section*: an explanation + the code that follows it | outputs and `pip install` lines dropped |
| Activity / guide PDFs | one page | PyMuPDF text in reading order; headers, footers, URLs, page counters removed |
| Image-only PDF (`module10_rl_slides.pdf`) | one slide | OCR with RapidOCR, cached; layout rebuilt so side-by-side boxes don't interleave |

Every document keeps metadata for citations: **file title, lecture number, slide/page/cell locator**.

> Run `uv run python -m study_assistant.ingest` once to (re)build `data/processed/documents.jsonl`. The cell below rebuilds it if it is missing.

In [ ]:
import pandas as pd
from study_assistant.config import COURSE_DIR, DOCUMENTS_PATH
from study_assistant.ingest import load_corpus, save_documents, load_documents

if not DOCUMENTS_PATH.exists():
    save_documents(load_corpus(COURSE_DIR))
docs = load_documents()

df = pd.DataFrame([{**d.to_dict(), **d.extra, "chars": len(d.text)} for d in docs])
df["lecture"] = df["lecture"].astype("Int64")
print(f"{len(df)} documents from {df.source_file.nunique()} files  (course dir: {COURSE_DIR.name})")
df[["doc_id", "source_type", "lecture", "section", "chars"]].head()

## 1. What did we load, per file?
Slide counts should equal the number of slides in each deck (this is also asserted in `tests/test_corpus.py`).

In [ ]:
per_file = (df.groupby("source_file")
              .agg(type=("source_type", "first"), lecture=("lecture", "first"), units=("doc_id", "count"),
                   chars=("chars", "sum"), low_content=("low_content", "sum"))
              .sort_values(["lecture", "type"], na_position="last"))
pd.set_option("display.max_rows", 100)
per_file

## 2. Coverage per lecture
Each lecture should have slides, and most have a practical notebook and/or activity. Lecture = `<NA>` are the agents / LLM add-on labs (RAG, LangGraph, evals, fine-tuning), which are not tied to one lecture deck.

Note the **lecture mapping**: practical notebooks are numbered differently from lectures (e.g. `Module_4_Practical_VAE.ipynb` is the practical for the **Lecture 5** VAE slides). `ingest/catalog.py` maps each notebook to the lecture whose content it covers, so citations point students to the right lecture.

In [ ]:
coverage = pd.crosstab(df["lecture"].fillna(-1).astype(int).replace(-1, "add-on"), df["source_type"], margins=True)
coverage

## 3. How long are the documents?
This tells us how to chunk in Step 2. The embedding model we plan to use (`all-MiniLM-L6-v2`) only reads the first **256 word-pieces** (~1,000 characters) of a text, so long units have to be split, or their tail would be invisible to retrieval.

In [ ]:
df.groupby("source_type")["chars"].describe(percentiles=[.5, .9]).round(0)

In [ ]:
long_units = df[df.chars > 1000]
print(f"{len(long_units)} of {len(df)} documents are longer than ~1000 chars and will need splitting in Step 2")
long_units.groupby("source_type").size()

## 4. Low-content documents
Title slides and section dividers (e.g. a slide that only says *"Variational Autoencoder"*) have no explanatory text. They stay in `documents.jsonl` but are flagged `low_content` so the index can skip them — otherwise they would out-rank the slides that actually explain the topic.

In [ ]:
df[df.low_content][["doc_id", "text"]].assign(text=lambda x: x.text.str.replace("\n", " / ").str[:70])

## 5. Spot checks — one example of each kind
Compare these against the original files to confirm nothing important is lost.

In [ ]:
def show(doc_id, n=900):
    d = next(d for d in docs if d.doc_id == doc_id)
    print(f"[{d.citation}]  lecture={d.lecture}  section={d.section!r}  extra={d.extra}\n")
    print(d.text[:n])

show("Module 5 - Modern Deep Learning Architectures Part 2.pptx::slide-16")

In [ ]:
show("Module_4_Practical_VAE.ipynb::section-6")

In [ ]:
show("Module 5 Activity - Adding VAE to Helper Library.pdf::page-1", 700)

In [ ]:
# OCR'd slide: the two callout boxes at the bottom are read one after the other, not interleaved
show("module10_rl_slides.pdf::slide-12")

## 6. Known limitations (to mention in the report)

1. **Equations and diagrams that are images are not captured.** e.g. Lecture 5 slide 16 says *"The final loss function that we use in training VAE takes the form:"* and the formula itself is a picture. `extra.n_images` records how many pictures a slide has. A possible later improvement is captioning those images with a vision model.
2. **OCR is not perfect**: subscripts/Greek letters in formulas come out approximately (`τ = (so,a0,r1,...)`), and small UI labels can be dropped.
3. **Code indentation in PDFs is lost** (PyMuPDF flattens it). Fine for retrieval and explanation; the notebooks keep the exact code.
4. **Notebook outputs are dropped** (training logs, printed tensors). They rarely explain concepts and would add noise.
5. **Students' own assignment submissions are deliberately excluded**: the assistant answers from what was *taught*.